In [ ]:
# standard python imports
import matplotlib.pyplot as plt
import numpy as np
import xarray as xr
import scipy as sp

# tidy3d imports
import tidy3d as td
import tidy3d.web as web
from tidy3d import material_library

web.configure(apikey="GoMnpjx6Gxf00bAc6wa2oPoqrR3mbWeYuuyi3aTPF94dR1kO")
td.config.logging_level = "ERROR"

API key configured successfully.


In [100]:
import tidy3d as td
import numpy as np

def build_slice_sim(wl_target=1.55, h_si=0.220, W_mmi=3.5, period_x=0.250, period_y=0.200, a=0.100, 
                     k_x=8.0, mode="TE"):
    """Builds the 1D slice sim for given parameters and specific modes."""

    freq0 = td.C_0 / wl_target
    fwidth = freq0 * 0.2

    padding_y=1.5
    padding_z=1.5

    mat_sio2 = material_library['SiO2']['Palik_Lossless']
    mat_si = material_library['cSi']['Palik_Lossless']

    geometry = [td.Structure(
        geometry=td.Box(center=(0, 0, 0), size=(td.inf, W_mmi, h_si)),
        medium=mat_si, name="MMI_Core"
    )]

    # Tile the holes along the width of the MMI (Y-axis)
    y_centers = np.arange(-W_mmi/2, W_mmi/2 + period_y, period_y)
    for y in y_centers:
        # Checkerboard pattern: shifted in X, tiled in Y
        geometry.append(td.Structure(
            geometry=td.Box(center=(-period_x/4, y, 0), size=(a, a, h_si+0.01)), medium=mat_sio2))
        geometry.append(td.Structure(
            geometry=td.Box(center=(period_x/4, y + period_y/2, 0), size=(a, a, h_si+0.01)), medium=mat_sio2))

    sources = []
    monitors = []
    np.random.seed(42)
    
    if "TE" in mode:
        sim_sym = (0, 0, 1)  # Z-PMC
        pol = "Ey"
    else:
        sim_sym = (0, 0, -1) # Z-PEC
        pol = "Ez"
    
    for i in range(15):
        x_pos = np.random.uniform(-period_x/2.1, period_x/2.1)
        y_pos = np.random.uniform(-W_mmi/2.1, W_mmi/2.1)
        phase = np.random.uniform(0, 2*np.pi)
        sources.append(td.PointDipole(
            center=(x_pos, y_pos, 0.05), source_time=td.GaussianPulse(freq0=freq0, fwidth=fwidth, phase=phase), 
            polarization=pol, name=f"dipole_{i}"))

    for i in range(5):
        x_pos = np.random.uniform(-period_x/2.1, period_x/2.1)
        y_pos = np.random.uniform(-W_mmi/2.1, W_mmi/2.1)
        monitors.append(td.FieldTimeMonitor(
            center=(x_pos, y_pos, 0.05), size=(0, 0, 0), interval=1, start=1e-12, name=f"mon_{i}"
        ))    

    return td.Simulation(
        size=(period_x, W_mmi + 2*padding_y, h_si + 2*padding_z),
        grid_spec=td.GridSpec.auto(min_steps_per_wvl=15),
        structures=geometry,
        sources=sources,
        monitors=monitors,
        run_time=10e-12,
        shutoff=0,
        # symmetry=sim_sym,
        boundary_spec=td.BoundarySpec(
            x=td.Boundary.bloch(bloch_vec=k_x),
            y=td.Boundary.pml(),
            z=td.Boundary.pml()
        ),
        medium=mat_sio2
    )

sim = build_slice_sim(mode="TE")
sim.plot_3d()

In [101]:
sim = build_slice_sim(mode="TM")
task_id = web.upload(sim, task_name="struct_test_6", folder_name="mmi_swg/")
estimate = web.estimate_cost(task_id)

23:07:17 SE Asia Standard Time Created task 'struct_test_6' with task_id        
                               'fdve-67a25df8-fc2b-4fb7-8839-cdd077fc4263' and  
                               task_type 'FDTD'.

                               View task using web UI at                        
                               ]8;id=290376;https://tidy3d.simulation.cloud/workbench?taskId=fdve-67a25df8-fc2b-4fb7-8839-cdd077fc4263\'https://tidy3d.simulation.cloud/workbench?]8;;\]8;id=136509;https://tidy3d.simulation.cloud/workbench?taskId=fdve-67a25df8-fc2b-4fb7-8839-cdd077fc4263\taskId]8;;\
                               ]8;id=290376;https://tidy3d.simulation.cloud/workbench?taskId=fdve-67a25df8-fc2b-4fb7-8839-cdd077fc4263\=]8;;\]8;id=104850;https://tidy3d.simulation.cloud/workbench?taskId=fdve-67a25df8-fc2b-4fb7-8839-cdd077fc4263\fdve]8;;\]8;id=290376;https://tidy3d.simulation.cloud/workbench?taskId=fdve-67a25df8-fc2b-4fb7-8839-cdd077fc4263\-67a25df8-fc2b-4fb7-8839-cdd077fc4263']8;;\.

                               Task folder: ]8;id=446597;https://tidy3d.simulation.cloud/folders/folder-b930ca1c-8bda-4228-b450-8401ef135c9b\'mmi_swg/']8;;\.

Output()

23:07:20 SE Asia Standard Time Maximum FlexCredit cost: 0.050. Minimum cost     
                               depends on task execution details. Use           
                               'web.real_cost(task_id)' to get the billed       
                               FlexCredit cost after a simulation run.

23:07:22 SE Asia Standard Time Maximum FlexCredit cost: 0.050. Minimum cost     
                               depends on task execution details. Use           
                               'web.real_cost(task_id)' to get the billed       
                               FlexCredit cost after a simulation run.

In [102]:
k_vals = np.linspace(4.0, 12.0, 11)
simulations_dict = {}

for mode in ["TE", "TM"]:
    for k in k_vals:
        sim_name = f"{mode}_kx_{k:.4f}"
        simulations_dict[sim_name] = build_slice_sim(k_x=k, mode=mode)

print(f"Submitting {len(simulations_dict)} simulations...")
batch = web.Batch(simulations=simulations_dict, folder_name="mmi_swg/slice_test/test_6/", verbose=True)
batch_results = batch.run(path_dir="data/mmi_swg/slice_test/test_6/")

Submitting 22 simulations...


Output()

23:08:54 SE Asia Standard Time Started working on Batch containing 22 tasks.

23:09:26 SE Asia Standard Time Maximum FlexCredit cost: 1.096 for the whole     
                               batch.

                               Use 'Batch.real_cost()' to get the billed        
                               FlexCredit cost after the Batch has completed.

Output()

23:10:48 SE Asia Standard Time Batch complete.

Output()

In [103]:
import scipy.interpolate as interp
from tidy3d.plugins.resonance import ResonanceFinder

# Constants
wl_target = 1.55
freq0 = td.C_0 / wl_target

# Standard Harminv window
res_finder = ResonanceFinder(freq_window=(freq0 * 0.7, freq0 * 1.3))

# Dictionary holding all 4 required modes
sweep_data = {
    "TE0": {"k": [], "f": []}, "TE1": {"k": [], "f": []},
    "TM0": {"k": [], "f": []}, "TM1": {"k": [], "f": []}
}
final_indices = {}

print("Extracting multi-mode resonances for TE and TM...")
print("-" * 50)

total_sims = len(batch_results)

for i, (sim_name, sim_data) in enumerate(batch_results.items(), 1):
    # Simulation names as "TE_kx_4.0000" or "TM_kx_4.0000"
    parts = sim_name.split("_")
    polarization = parts[0]  # Extracts "TE" or "TM"
    k_x_current = float(parts[2])
    
    print(f"[{i}/{total_sims}] Analyzing {sim_name}...")
    
    # Grab all 5 random monitors at once
    time_signals = [sim_data[f"mon_{j}"] for j in range(5)]
    df_res = res_finder.run(signals=time_signals).to_dataframe().reset_index()
    
    if not df_res.empty:
        # Bandpass Filter (150 - 230 THz)
        df_clean = df_res[(df_res['freq'] > 150e12) & (df_res['freq'] < 230e12)]
        
        if len(df_clean) >= 2:
            # Sort by lowest frequency (highest n_eff)
            df_sorted = df_clean.sort_values(by="freq", ascending=True)
            
            f_mode0 = df_sorted.iloc[0]["freq"]
            f_mode1 = df_sorted.iloc[1]["freq"]
            
            sweep_data[f"{polarization}0"]["k"].append(k_x_current)
            sweep_data[f"{polarization}0"]["f"].append(f_mode0)
            
            sweep_data[f"{polarization}1"]["k"].append(k_x_current)
            sweep_data[f"{polarization}1"]["f"].append(f_mode1)
            
            print(f"        -> {polarization}0: {f_mode0/1e12:.2f} THz | {polarization}1: {f_mode1/1e12:.2f} THz")
            
        elif len(df_clean) == 1:
            print(f"        -> Only 1 mode found in window ({df_clean.iloc[0]['freq']/1e12:.2f} THz). Skipping.")
        else:
            print("        -> No modes found in window (Likely a Bandgap).")
    else:
        print("        -> No resonances found (Bandgap).")

print("\n" + "="*50)
print("🎯 EXACT EFFECTIVE INDICES AT 1550nm 🎯")
print("="*50)

for mode in ["TE0", "TE1", "TM0", "TM1"]:
    f_arr = sweep_data[mode]["f"]
    k_arr = sweep_data[mode]["k"]
    
    if len(f_arr) > 1:
        # Sort arrays to prevent interpolation errors
        sort_idx = np.argsort(f_arr)
        f_arr = np.array(f_arr)[sort_idx]
        k_arr = np.array(k_arr)[sort_idx]
        
        interp_func = interp.interp1d(f_arr, k_arr, kind='linear', fill_value="extrapolate")
        exact_kx = interp_func(freq0)
        
        n_eff = (exact_kx * td.C_0) / (2 * np.pi * freq0)
        final_indices[mode] = float(n_eff)
        
        print(f"{mode}: kx = {exact_kx:.4f} rad/um  -->  n_eff = {n_eff:.5f}")
    else:
        print(f"{mode}: Interpolation failed. Not enough clean data points.")

print("\n" + "="*50)
print("📏 MMI BEAT LENGTH RESULTS 📏")
print("="*50)

# Calculate L_MMI for TE
if "TE0" in final_indices and "TE1" in final_indices:
    L_mmi_TE = (3 * wl_target) / (4 * (final_indices["TE0"] - final_indices["TE1"]))
    print(f"L_MMI (TE): {L_mmi_TE:.3f} um")
else:
    L_mmi_TE = None
    print("[!] Missing TE mode data.")

# Calculate L_MMI for TM
if "TM0" in final_indices and "TM1" in final_indices:
    L_mmi_TM = (3 * wl_target) / (4 * (final_indices["TM0"] - final_indices["TM1"]))
    print(f"L_MMI (TM): {L_mmi_TM:.3f} um")
else:
    L_mmi_TM = None
    print("[!] Missing TM mode data.")

# Calculate Polarization Mismatch
if L_mmi_TE is not None and L_mmi_TM is not None:
    print("-" * 50)
    mismatch = abs(L_mmi_TE - L_mmi_TM)
    print(f"ΔL Mismatch: {mismatch:.3f} um")

Extracting multi-mode resonances for TE and TM...
--------------------------------------------------
[1/22] Analyzing TE_kx_4.0000...
        -> No modes found in window (Likely a Bandgap).
[2/22] Analyzing TE_kx_4.8000...
        -> TE0: 153.47 THz | TE1: 153.62 THz
[3/22] Analyzing TE_kx_5.6000...
        -> Only 1 mode found in window (229.90 THz). Skipping.
[4/22] Analyzing TE_kx_6.4000...
        -> Only 1 mode found in window (229.90 THz). Skipping.
[5/22] Analyzing TE_kx_7.2000...
        -> TE0: 153.47 THz | TE1: 153.62 THz
[6/22] Analyzing TE_kx_8.0000...
        -> No modes found in window (Likely a Bandgap).
[7/22] Analyzing TE_kx_8.8000...
        -> TE0: 153.47 THz | TE1: 153.62 THz
[8/22] Analyzing TE_kx_9.6000...
        -> Only 1 mode found in window (229.90 THz). Skipping.
[9/22] Analyzing TE_kx_10.4000...
        -> Only 1 mode found in window (229.90 THz). Skipping.
[10/22] Analyzing TE_kx_11.2000...
        -> TE0: 153.47 THz | TE1: 153.62 THz
[11/22] Analyzing TE_k

c:\Users\irfan\AppData\Local\Programs\Python\Python313\Lib\site-packages\scipy\interpolate\_interpolate.py:497: RuntimeWarning: divide by zero encountered in divide
  slope = (y_hi - y_lo) / (x_hi - x_lo)[:, None]
